# REE Extraction Basics: Using the difflow_ree Module

This notebook demonstrates the **difflow_ree** plugin for rare earth element (REE) solvent extraction.

## Background

The difflow_ree module provides:
- **Database** of 10 REE elements with properties
- **4 industrial extractants** (D2EHPA, PC88A, Cyanex272, TBP)
- **pH-dependent distribution models**
- **Unit operations** for extraction, scrubbing, stripping
- **Pre-built flowsheet templates**

## What You'll Learn

1. Access REE element and extractant databases
2. Calculate pH-dependent distribution coefficients
3. Simulate multi-stage extraction units
4. Perform sensitivity analysis and optimization
5. Use automatic differentiation for gradients

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad

# Enable 64-bit precision for numerical stability
jax.config.update("jax_enable_x64", True)

# Import difflow_ree components
from difflow_ree import (
    # Database access
    get_element,
    get_extractant,
    list_ree_elements,
    list_extractants,
    # Distribution model
    REEDistribution,
    # Unit operations
    REEExtractor,
    REEExtractorParams,
)

# Import stream utilities
from difflow.streams import make_stream, get_flows

## 1. Exploring the REE Database

The difflow_ree module includes a comprehensive database of REE properties.

In [ ]:
# List all available REE elements
elements = list_ree_elements()
print(f"Available REE elements ({len(elements)}):")
print("  " + ", ".join(elements))

# Get detailed properties for specific elements
print("\nElement Properties:")
print("="*70)
print(f"{'Symbol':<8} {'Name':<15} {'Group':<10} {'Price ($/kg)':<15}")
print("-"*70)

for symbol in ["La", "Nd", "Eu", "Dy"]:
    elem = get_element(symbol)
    print(f"{elem.symbol:<8} {elem.name:<15} {elem.group:<10} ${elem.price_usd_kg:<14.2f}")

## 2. Exploring Extractant Database

Industrial extractants for REE separation.

In [ ]:
# List available extractants
extractants = list_extractants()
print(f"Available extractants ({len(extractants)}):")
for ext_name in extractants:
    print(f"  • {ext_name}")

# Get D2EHPA properties
d2ehpa = get_extractant("D2EHPA")
print(f"\nD2EHPA Properties:")
print(f"  Full name: {d2ehpa.full_name}")
print(f"  Formula: {d2ehpa.formula}")
print(f"  MW: {d2ehpa.molecular_weight:.2f} g/mol")
print(f"  pKa: {d2ehpa.pKa}")
print(f"  Type: {d2ehpa.extractant_type}")
print(f"  Typical concentration: {d2ehpa.typical_concentration} M")
print(f"  Valid pH range: {d2ehpa.valid_ph_range}")
print(f"  Cost: ${d2ehpa.cost_usd_kg}/kg")

## 3. Distribution Coefficient Calculations

Distribution coefficients (D) determine how REEs partition between aqueous and organic phases:

$$D = \frac{[\text{REE}]_{\text{organic}}}{[\text{REE}]_{\text{aqueous}}}$$

The model includes:
- pH dependence: $\log_{10}(D) = a + b \cdot pH + c \cdot pH^2$
- Temperature correction
- Extractant concentration effects

In [ ]:
# Create distribution model for D2EHPA
dist = REEDistribution(
    extractant="D2EHPA",
    elements=("La", "Nd", "Dy"),
    concentration=0.5,  # 0.5 M in organic phase
)

# Calculate D values at pH 3.0
D_values = dist.get_D_all(pH=3.0, T=298.15)

print("Distribution Coefficients at pH 3.0, 25°C:")
print("="*50)
for elem, D in D_values.items():
    print(f"  D({elem}) = {float(D):8.4f}")

# Calculate separation factors
SF_Nd_La = float(D_values["Nd"] / D_values["La"])
SF_Dy_Nd = float(D_values["Dy"] / D_values["Nd"])

print(f"\nSeparation Factors:")
print(f"  SF(Nd/La) = {SF_Nd_La:.2f}")
print(f"  SF(Dy/Nd) = {SF_Dy_Nd:.2f}")
print("\n💡 Higher SF = easier separation")

## 4. pH Effect on Distribution

D values increase exponentially with pH for acidic extractants.

In [ ]:
# Effect of pH on distribution coefficients
pH_values = [2.0, 2.5, 3.0, 3.5, 4.0]

print("pH Effect on Distribution Coefficients:")
print("="*60)
print(f"{'pH':<6} {'D(La)':<12} {'D(Nd)':<12} {'D(Dy)':<12}")
print("-"*60)

for pH in pH_values:
    D_vals = dist.get_D_all(pH=pH, T=298.15)
    print(f"{pH:<6.1f} {float(D_vals['La']):<12.4f} {float(D_vals['Nd']):<12.4f} {float(D_vals['Dy']):<12.4f}")

print("\n📊 Key insight: D increases ~10x per pH unit for acidic extractants")

## 5. Multi-Stage Extraction Unit

Simulate a counter-current extraction cascade.

In [ ]:
# Create extraction unit parameters
params = REEExtractorParams(
    n_stages=5,
    extractant="D2EHPA",
    elements=("La", "Nd", "Dy"),
    pH=3.0,
    stage_efficiency=0.85,  # 85% approach to equilibrium per stage
)

# Create extractor
extractor = REEExtractor(params)

# Define feed streams
feed = make_stream(
    flows={
        "H2O": 10.0,
        "La": 0.01,
        "Nd": 0.02,
        "Dy": 0.01,
    },
    T=298.15,
    P=101325.0,
)

solvent = make_stream(
    flows={
        "Organic": 8.0,
        "La": 0.0,
        "Nd": 0.0,
        "Dy": 0.0,
    },
    T=298.15,
    P=101325.0,
)

# Run extraction
raffinate, extract, info = extractor(feed, solvent)

# Analyze results
feed_flows = get_flows(feed)
raff_flows = get_flows(raffinate)
ext_flows = get_flows(extract)

print("Extraction Results (5 stages, pH 3.0):")
print("="*70)
print(f"{'Element':<10} {'Feed':<12} {'Raffinate':<12} {'Extract':<12} {'Recovery %':<12}")
print("-"*70)

for elem in ["La", "Nd", "Dy"]:
    feed_val = float(feed_flows[elem])
    raff_val = float(raff_flows[elem])
    ext_val = float(ext_flows[elem])
    recovery = (ext_val / feed_val) * 100
    
    print(f"{elem:<10} {feed_val:<12.4f} {raff_val:<12.4f} {ext_val:<12.4f} {recovery:<12.1f}")

# Calculate purities
total_REE_extract = sum(float(ext_flows[e]) for e in ["La", "Nd", "Dy"])
nd_purity = float(ext_flows["Nd"]) / total_REE_extract * 100

print(f"\nExtract Composition:")
print(f"  Nd purity (among REEs): {nd_purity:.1f}%")

## 6. Automatic Differentiation: Gradients

Compute exact gradients for sensitivity analysis and optimization.

In [ ]:
def nd_recovery(pH):
    """Calculate Nd recovery as a function of pH."""
    dist_pH = REEDistribution(
        extractant="D2EHPA",
        elements=("La", "Nd", "Dy"),
        concentration=0.5,
    )
    
    D_val = dist_pH.get_D("Nd", pH, T=298.15)
    
    # Simplified extraction efficiency model
    # For counter-current: E ≈ 1 - 1/(1 + D*S/F)^N
    S_F = 0.8  # Solvent/Feed ratio
    N = 5.0    # Number of stages
    
    extraction_factor = D_val * S_F
    recovery = 1.0 - 1.0 / (1.0 + extraction_factor)**N
    
    return recovery

# Compute gradient
pH_operating = 3.0
dR_dpH = grad(nd_recovery)(pH_operating)

print("Sensitivity Analysis:")
print("="*50)
print(f"Operating pH: {pH_operating}")
print(f"Nd recovery: {float(nd_recovery(pH_operating))*100:.2f}%")
print(f"\n∂(Nd recovery)/∂pH = {float(dR_dpH):.4f}")
print(f"\n💡 Interpretation:")
print(f"  • Increasing pH by 0.1 units increases recovery by ~{float(dR_dpH)*0.1*100:.2f}%")
print(f"  • pH control is critical for consistent performance")

## 7. Optimization: Find Optimal pH

Use gradient descent to maximize Nd purity.

In [ ]:
def nd_purity_objective(pH):
    """Nd purity among all REEs in extract."""
    dist_pH = REEDistribution(
        extractant="D2EHPA",
        elements=("La", "Nd", "Dy"),
        concentration=0.5,
    )
    
    D_vals = dist_pH.get_D_all(pH, T=298.15)
    
    # Assume single equilibrium stage for simplicity
    # In extract: [REE] ∝ D * [REE]_feed
    feed_ratio = {"La": 0.01, "Nd": 0.02, "Dy": 0.01}
    
    extract_amounts = {
        elem: D_vals[elem] * feed_ratio[elem]
        for elem in ["La", "Nd", "Dy"]
    }
    
    total = sum(extract_amounts.values())
    purity = extract_amounts["Nd"] / (total + 1e-10)
    
    return purity

# Gradient ascent (maximize purity)
pH_opt = 3.0
learning_rate = 0.1

print("Optimization Progress:")
print("="*50)
print(f"{'Iter':<8} {'pH':<10} {'Nd Purity %':<15}")
print("-"*50)

for i in range(20):
    gradient = grad(nd_purity_objective)(pH_opt)
    pH_opt = pH_opt + learning_rate * gradient
    
    # Constrain to valid range
    pH_opt = jnp.clip(pH_opt, 1.0, 5.0)
    
    if (i + 1) % 5 == 0:
        purity = nd_purity_objective(pH_opt)
        print(f"{i+1:<8} {float(pH_opt):<10.3f} {float(purity)*100:<15.2f}")

final_purity = nd_purity_objective(pH_opt)
print(f"\n✓ Optimal pH: {float(pH_opt):.3f}")
print(f"  Nd purity: {float(final_purity)*100:.2f}%")

## 8. Comparing Different Extractants

Compare D2EHPA, PC88A, and Cyanex272 for Nd extraction.

In [ ]:
extractant_list = ["D2EHPA", "PC88A", "Cyanex272"]
pH_compare = 3.0

print(f"Extractant Comparison at pH {pH_compare}:")
print("="*70)
print(f"{'Extractant':<15} {'D(Nd)':<12} {'SF(Nd/La)':<12} {'SF(Dy/Nd)':<12}")
print("-"*70)

for ext_name in extractant_list:
    dist_compare = REEDistribution(
        extractant=ext_name,
        elements=("La", "Nd", "Dy"),
        concentration=0.5,
    )
    
    D_vals = dist_compare.get_D_all(pH=pH_compare, T=298.15)
    
    D_Nd = float(D_vals["Nd"])
    SF_Nd_La = float(D_vals["Nd"] / D_vals["La"])
    SF_Dy_Nd = float(D_vals["Dy"] / D_vals["Nd"])
    
    print(f"{ext_name:<15} {D_Nd:<12.4f} {SF_Nd_La:<12.2f} {SF_Dy_Nd:<12.2f}")

print("\n💡 Insights:")
print("  • PC88A provides better light REE selectivity")
print("  • D2EHPA is the most cost-effective for general use")
print("  • Cyanex272 operates at higher pH (4-6) for specialty separations")

## Summary

This notebook demonstrated:

1. **Database access** - REE properties and extractant data
2. **Distribution models** - pH-dependent D values
3. **Multi-stage extraction** - Counter-current cascades
4. **Automatic differentiation** - Exact gradients for sensitivity
5. **Optimization** - Gradient-based parameter tuning
6. **Extractant comparison** - Selecting best extractant for the job

**Next Steps:**
- See `21_custom_extractants.ipynb` to learn how to define custom extractants
- Explore pre-built flowsheet templates for complete separation trains
- Add economic analysis to evaluate process profitability